In [5]:
from config import CONTENT_DIR, GOOGLE_AI_API_KEY
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyPDFium2Loader
from langchain_community.document_loaders.parsers import RapidOCRBlobParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
# 1. 3072차원 설정
emb_3072 = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2', 
    api_key=GOOGLE_AI_API_KEY
)
# 2. 256차원 설정
emb_256 = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2', 
    output_dimensionality=256,
    api_key=GOOGLE_AI_API_KEY
)
sentences = [
    "마법사는 마나를 다루는 직업이다.",       # A
    "위자드는 마법 에너지로 싸우는 캐릭터다.", # B (A와 유사)
    "오늘 점심에 김치찌개를 먹어야지"         # C (무관함)
]
# 벡터 생성
vecs_3072 = emb_3072.embed_documents(sentences)
vecs_256 = emb_256.embed_documents(sentences)
# 코사인 유사도 계산 함수
def print_similarity(title, vecs):
    sim_AB = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
    sim_AC = cosine_similarity([vecs[0]], [vecs[2]])[0][0]
    print(f"[{title}]")
    print(f" - 마법사 vs 위자드 (유사): {sim_AB:.4f}")
    print(f" - 마법사 vs 김치찌개 (무관): {sim_AC:.4f}")
    print(f" - 판별 격차 (유사 - 무관): {sim_AB - sim_AC:.4f}\n")
print_similarity("3072차원", vecs_3072)
print_similarity("256차원", vecs_256)

[3072차원]
 - 마법사 vs 위자드 (유사): 0.7442
 - 마법사 vs 김치찌개 (무관): 0.4361
 - 판별 격차 (유사 - 무관): 0.3082

[256차원]
 - 마법사 vs 위자드 (유사): 0.7708
 - 마법사 vs 김치찌개 (무관): 0.4847
 - 판별 격차 (유사 - 무관): 0.2860



In [7]:
docs = [
    "신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.",
    "법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.",
    "개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.",
    "장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.",
    "재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.",
]

doc_vectors = emb_256.embed_documents(docs)

question = "법인카드 영수증은 언제까지 제출해야 하나요?"
q_vector = emb_256.embed_query(question)

# 질문 vs 각 문서 유사도
# doc_vectors는 이미 2차원 배열이므로, q_vector만 2차원([[q_vector]])으로 만들어 한 번에 연산
scores = cosine_similarity([q_vector], doc_vectors)[0]
ranked = sorted(zip(scores, docs), reverse=True)

print(f"질문: {question}\n")
print("=== 유사도 순위 ===")
for score, doc in ranked:
    print(f"  {score:.3f}  {doc}")

질문: 법인카드 영수증은 언제까지 제출해야 하나요?

=== 유사도 순위 ===
  0.812  법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다.
  0.589  재택근무 신청은 최소 하루 전까지 근태 시스템에서 등록해야 하며, 팀장의 승인을 받은 뒤 근무 장소를 명시해야 한다.
  0.588  신규 입사자는 입사 후 7일 이내에 보안 교육을 이수해야 하며, 교육 미이수 시 사내 시스템 접근 권한이 제한될 수 있다.
  0.575  개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 하며, 고객명과 연락처는 마스킹 대상에 포함된다.
  0.560  장애 보고서는 발생 시각, 영향 범위, 임시 조치, 재발 방지 대책을 포함하여 서비스 복구 후 24시간 이내에 작성해야 한다.


In [17]:
loader = TextLoader(CONTENT_DIR / "company_docs/hr_policy.txt", encoding='utf-8')
doc = loader.load()

print(doc[0].page_content[:50])

# [인사 정책 매뉴얼] 즐겁고 공정한 직장 문화를 위한 가이드

본 안내서는 우리 회사의


In [ ]:
def load_all_documents(data_dir):
    """폴더에서 .txt와 .md 파일을 스캔하여 Document 목록으로 로드합니다."""
    path = Path(data_dir)
    
    # 1. 한 번의 디렉터리 스캔으로 두 확장자를 가진 파일들을 모두 리스트업합니다.
    file_paths = [p for p in path.rglob("*") if p.suffix in [".txt", ".md"]]
    
    # 2. 각 파일에 대해 TextLoader를 호출하여 문서를 로드합니다.
    all_docs = []
    for fp in file_paths:
        loader = TextLoader(str(fp), encoding="utf-8")
        all_docs.extend(loader.load())
        
    return all_docs

In [3]:
filename = Path(CONTENT_DIR / "[이슈리포트 2022-2호] 혁신성장 정책금융 동향.pdf")

loader = PyPDFLoader(str(filename), mode='page', images_inner_format='markdown-img', images_parser=RapidOCRBlobParser())
pages = loader.load()

print(pages[5].page_content)

[INFO] 2026-06-15 12:42:57,129 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-15 12:42:57,197 [RapidOCR] download_file.py:60: File exists and is valid: /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-15 12:42:57,198 [RapidOCR] main.py:65: Using /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-15 12:42:57,285 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-15 12:42:57,287 [RapidOCR] download_file.py:60: File exists and is valid: /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-15 12:42:57,288 [RapidOCR] main.py:65: Using /home/hong/project/ai-camp-note/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-15 12:42:57,341 [RapidOCR] base.py:22: Using engine_nam

| 6 | CIS이슈리포트 2022-2호 
▶(주요품목① : 5G 이동통신) 정보통신 테마 내 기술분야 중 혁신성장 정책금융 공급규모가 가장 큰 차세대무선통신미디어 분야의 경우 4G/5G 기술품목의 정책금융 공급 비중이 가장 높은 것으로 확인됨[차세대무선통신미디어 분야 내 기술품목별 혁신성장 정책금융 공급액 추이](단위: 억 원)
▶5G 이동통신 시스템은 ITU(International Telecommunication Union)가 정의한 5세대 이동통신 규격을 만족시키는 무선 이동통신 네트워크 기술로, 2019년부터 국내 서비스를 시작함￮4G 이동통신 시스템(LTE)과 비교할 때 전송속도의 향상(1Gbps→20Gbps), 이동성 향상(350km/h→500km/h에서 끊김없는 데이터 전송 가능), 최대 연결가능 기기수 증가(10만 대 →100만 대 이상), 데이터 전송지연 감소(10ms→1ms) 등의 향상된 기능을 제공함￮5G는 전송속도 향상, 다수기기 접속 및 지연시간 단축을 위해 ①밀리미터파 통신이 가능한 주파수 확장, ②스몰셀(Small cell)을 도입한 기지국, ③다중안테나 송수신(Massive MIMO), ④네트워크 슬라이싱(Network Slicing) 등의 기술을 도입함[5G 주요 요소기술 특징]
자료: 삼정 KPMG



![25,000
20,000
15,000
10,000
RO
5,000
2017
2018
2019
2020
2021
4G/5G
(loT,M2M)
■吾号](#)
![MassiveMMO+
4G5G
（）
4G
2.6Ghz
4G
5G
品号
3.5Ghz
28Ghz
（m）
5G
（1k0H）
loboly
Massive
（km）
loT
5G3.5Ghz28Ghz0
MassiveMIMOEI
舍](#)
![Korea
Credit Information
Services](#)


In [4]:
loader = PyPDFium2Loader(str(filename), mode='page')
pages = loader.load()

print(pages[3].page_content)

| 4 | CIS이슈리포트 2022-2호 
[혁신성장 ICT 산업 정책금융 공급 현황]
 (단위: 억 원, 괄호는 점유율 %)
구분 2017년 말 2018년 말 2019년 말 2020년 말 2021년 말
혁신성장 ICT 산업
45,075 72,799 81,805 139,687 169,089 
(18.7) (20.7) (18.5) (20.3) (19.8)
정보통신
15,658 27,417 39,033 65,324 77,750 
(6.5) (7.8) (8.8) (9.5) (9.1)
전기전자
26,637 38,521 35,922 62,856 77,485 
(11.1) (10.9) (8.1) (9.1) (9.1)
센서측정
2,780 6,861 6,851 11,506 13,854 
(1.2) (1.9) (1.5) (1.7) (1.6)
혁신성장 정책금융 
총 공급액 240,787 351,987 443,180 688,409 854,338 
3. 정보통신 테마 혁신성장 정책금융 현황 및 관련 산업 동향
▶ (지원 현황) 정보통신 테마를 구성하는 기술분야별 정책금융 지원 현황 분석결과, 공급점유율 관점
에서는 차세대무선통신미디어 분야에 가장 많은 정책자금이 투입 되고 있으며, 공급량 증가율 관점
에서는 능동형컴퓨팅 분야로의 정책자금 지원 증가 속도가 가장 빠른 추세임
￮ 차세대무선통신미디어란 전송속도 향상, 소모전력 절감, 고속이동 중 끊김없는 통신 등 새로운 무선
환경에 필요한 통신, 인프라 및 서비스 기술을 통칭하며, 4G/5G/6G, 사물인터넷, 방송통신인프라 
등의 품목으로 구성됨
- 정보통신 테마 내 혁신성장 정책금융 공급 규모의 약 50%를 점유하고 있으며, 이는 초연결 미래
사회를 구축하기 위해 네트워크 기반 기술 사업화에 대한 정책자금 공급이 꾸준함에 따른 것으로 
분석됨
￮ 능동형컴퓨팅이란 거대하고 복잡해지는 데이터의 효율적 가공과 관리를 위한 인간두뇌와 유사한 
형태의 정보처리기술을 말하며, 인공지능, 상황인지컴퓨팅 등의 품목으로 구성됨
- 컴퓨팅 기술을 활

In [6]:
SAMPLE_DOC = """## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.

### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.

### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,             # 한 청크 목표 크기
    chunk_overlap=60,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_text(SAMPLE_DOC)
chunks

['## 회사 업무 운영 가이드\n\n### 1. 신규 입사자 온보딩\n신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.',
 '### 2. 재택근무 신청\n재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.',
 '### 3. 법인카드 및 경비 처리\n법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.',
 '### 4. 보안 및 개인정보 보호\n개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.']